# Stage 6 Materials Proxy: Fluence And Thresholds

This notebook computes optical fluence and threshold-comparison
planning metrics from simulated scalar/vortex beam cases.

These material-facing results are planning proxies unless explicitly
marked experimentally_calibrated. A thresholded fluence map is not a
calibrated prediction of ablation, void formation, refractive-index
change, or weld success.


## Stage 8.7 Adjustable Quick-Look Guidance

<!-- STAGE87: adjustable quicklook guidance -->

For fast parameter scouting, use `notebooks/quicklook/00_quick_beam_to_sample_simulator.ipynb`. This notebook remains on its locked stage path: existing execution logic, propagation-power labels, material-proxy caveats, and governance routing are unchanged.

Safe local edits are the explicit config variables already exposed by this notebook, or a copied exploratory run. Keep `fail` and `marginal` labels visible. If a displayed image is visually smoothed, treat that as display interpolation only; rerun balanced/publication sampling before numerical interpretation.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

import bessel_twin_core as bt
from Publication_Study import publication_diagnostics as pdiag
from vbb_study import setup_study, vbb_materials
from vbb_study.publication import materials as material_schema

PATHS = setup_study.bootstrap(Path.cwd())
CSV_OUT = PATHS["csv"] / "materials"
CSV_OUT.mkdir(parents=True, exist_ok=True)


## Editable Notebook Controls

<!-- STAGE88: editable controls -->

This cell exposes the intended user-editable controls for exploratory runs. The locked stage logic below is preserved: changing these controls is for local investigation unless the notebook explicitly wires a value into a regenerated canonical output. Keep QA, caveats, and fail/marginal labels visible. For fast beam-to-sample exploration use the quicklook notebook; for publication-grade outputs use the locked stage runner.


In [ ]:
# STAGE88: visible editable controls for exploratory notebook use.
# Edit NOTEBOOK_CONTROLS below and re-run this cell to apply material
# parameter overrides for downstream analysis cells.
from vbb_study.publication import notebook_controls as nb_controls

NOTEBOOK_CONTROLS = nb_controls.make_notebook_controls(
    stage='materials',
    # ── edit these for materials analysis ───────────────────────────────────
    pulse_energy_uJ=10.0,
    threshold_fluence_J_cm2=1.0,
    pulse_count=1,
)

# Wire control parameters to named variables used by downstream cells.
_p = NOTEBOOK_CONTROLS.parameters or {}
PULSE_ENERGY_uJ = float(_p.get("pulse_energy_uJ", 10.0))
THRESHOLD_FLUENCE_J_CM2 = float(_p.get("threshold_fluence_J_cm2", 1.0))
PULSE_COUNT = int(_p.get("pulse_count", 1))

try:
    display(nb_controls.describe_controls(NOTEBOOK_CONTROLS))
except NameError:
    print(nb_controls.describe_controls(NOTEBOOK_CONTROLS).to_string(index=False))


In [ ]:
# Interactive beam quicklook — adjust sliders and click "Update plots".
# This shows beam propagation for the configured parameters.
# Runs a fast preview only; nothing is saved and this is independent of the
# material analysis cells below.
from dataclasses import replace
from vbb_study.publication import notebook_widgets as nbw
from vbb_study.config import um as _um

_ql_base = bt.default_config("fast")
_panel = nbw.interactive_quicklook(_ql_base, method='holographic', preset='fast')
display(_panel)


## Optical Fluence Versus Material Response

The optical field supplies intensity, pulse energy, propagation QA,
and Bessel-zone metrics. The material layer only compares that
optical fluence with configured threshold proxies. No row here is
experimentally calibrated.


In [2]:
summary, cases = vbb_materials.build_shortlist_design_table(
    pdiag.DEFAULT_SHORTLIST,
    preset="fast",
    path="realistic",
)
summary = material_schema.ordered_material_frame(summary.to_dict("records"))

route_notes = {
    "scalar_bessel": "ell=0 scalar Bessel optical field",
    "vortex_bessel": "ell>0 vortex Bessel optical field",
    "vector": "vector optical route can be joined later through shared optical metrics",
}
summary["source_optical_route"] = summary["beam_family"].map(route_notes).fillna(summary["beam_family"])
summary["output_category"] = "planning_proxy"
summary["safe_for_design_comparison"] = True

summary = material_schema.ordered_material_frame(summary.to_dict("records"))
summary_path = CSV_OUT / "material_proxy_fluence_threshold_summary.csv"
summary.to_csv(summary_path, index=False)

compatibility_path = CSV_OUT / "07_materials_design_table.csv"
summary.to_csv(compatibility_path, index=False)

display_cols = [
    "case_id",
    "beam_family",
    "material_model_status",
    "calibration_status",
    "threshold_source",
    "peak_fluence_J_cm2",
    "fluence_to_threshold_ratio",
    "thresholded_area_um2",
    "xz_energy_conservation_status",
]
display(summary[display_cols])
print(summary_path)
print(compatibility_path)


,case_id,beam_family,material_model_status,calibration_status,threshold_source,peak_fluence_J_cm2,fluence_to_threshold_ratio,thresholded_area_um2,xz_energy_conservation_status
0,ell0_core3_L150,scalar_bessel,planning_proxy,uncalibrated,configured_placeholder,25.191875,31.374000,44.620072,normalised_visualisation
1,ell3_core3_L150,vortex_bessel,planning_proxy,uncalibrated,configured_placeholder,4.718381,5.876279,33.701723,normalised_visualisation
2,ell5_core4_L200,vortex_bessel,planning_proxy,uncalibrated,configured_placeholder,7.667883,9.549594,129.694835,normalised_visualisation


C:\PhD\Code\Publication_Study\outputs\csv\materials\material_proxy_fluence_threshold_summary.csv
C:\PhD\Code\Publication_Study\outputs\csv\materials\07_materials_design_table.csv


## Reading The Proxy

The threshold columns are safe for relative planning comparison
between optical cases. They are not material-response predictions.
Line-fluence XZ maps are diagnostic planning visualisations unless
explicitly generated from an energy-conserving 3D deposition model.
